In [21]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["cb03__age_imputed_title_Pclass_and_bins"]

# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
#     result_df = save_results(exp_result)
#     save_configs(exp_config)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="baseline__raw",
#             compare_groups=[Name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass
Experiment: fe08__fare_per_family_member
Experiment: fe09__ticket_group_size
Experiment: fe10__fare_per_ticket_member
Experiment: fe11__age_bin
Experiment: fe12__sex_pclass
Experiment: cb01__age_and_bins
Experiment: cb02__age_imputed_title_and_bins
Experiment: cb03__age_imputed_title_Pclass_and_bins
Running baseline__raw experiments...
running exp: {'name': 'baseline__raw__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], 'feature_engineering': [], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_

In [23]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [24]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

running exp: {'name': 'cb03__age_imputed_title_Pclass_and_bins__logreg', 'features': ['Pclass', 'Sex', 'SibSp', 'Parch', 'Embarked', 'Age', 'Age_bin'], 'feature_engineering': [<function age_imputed_by_title_pclass at 0x0000028FD1B009D0>, <function add_age_bin at 0x0000028FD1B00CA0>], 'preprocessing': {'numeric_features': ['SibSp', 'Parch', 'Age'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass', 'Age_bin'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Combo 03: Exploring the effect of using both raw age and age bins.', 'stage': 'cb03', 'feature_group': 'age_imputed_title_Pclass_and_bins', 'group': 'cb03__age_imputed_title_Pclass_and_bins', 'domain': 'age'}
running exp: {'name': 'cb

In [25]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()

# For combos:
all_results = load_results()
references = ['cb01__age_and_bins','cb02__age_imputed_title_and_bins']
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


Full workflow report:

Report
### cb03__age_imputed_title_Pclass_and_bins

_Description pending._

<details>
<summary>Conclusion</summary>


#### Interpretation

- Verdict: model_specific_positive
- Recommended for specific models:
  - logreg: test_accuracy_mean: 0.016
    - Secondary gains:
      - test_f1_mean: 0.017
  - svc: test_accuracy_mean: 0.004
  - random_forest: test_accuracy_mean: 0.006
  - extra_trees: test_accuracy_mean: 0.013
    - Secondary gains:
      - test_f1_mean: 0.02
- Notable secondary improvements in non-recommended models:
  - decision_tree: test_f1_mean: 0.012


#### Conclusion

_Conclusion pending._

</details>

<details>
<summary>Experiment details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group                           | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:

In [26]:
# import pprint
# Feature_effect = analyze_feature_effect(workflow['comparison'])
# pprint.pprint(Feature_effect)

In [27]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [28]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [29]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [30]:
# print(workflow["all_results"])

In [31]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

In [32]:
# bins = [0, 14, 35, 60, 100]
# labels = ['0', '2', '3', '1']
# exp_df['Age_bin'] = pd.cut(exp_df['Age'], bins=bins, labels=labels, right=False)

In [33]:
# for age in range(0,18):
#     test_df = exp_df[exp_df['Age'] == age][['Age','Survived']]
#     test_df = test_df.groupby('Survived').value_counts()
#     print(test_df)

In [34]:
def domain_best_by_model(
    results_df,
    domain,
    metric="test_accuracy_mean",
    include_baseline=True,
    only_success=True,
):
    df = results_df.copy()

    if only_success and "status" in df.columns:
        df = df[df["status"] == "success"]

    if metric not in df.columns:
        raise ValueError(f"Metric '{metric}' not found in results dataframe.")

    domain_df = df[df["domain"] == domain]

    if include_baseline:
        baseline_df = df[df["group"] == "baseline__raw"]
        domain_df = pd.concat([baseline_df, domain_df], ignore_index=True)

    if domain_df.empty:
        raise ValueError(f"No results found for domain: {domain}")

    best_rows = (
        domain_df
        .sort_values(metric, ascending=False)
        .groupby("model_name", as_index=False)
        .first()
    )

    columns = [
        "model_name",
        "experiment",
        "group",
        "domain",
        metric,
        "test_f1_mean",
    ]

    columns = [col for col in columns if col in best_rows.columns]

    return (
        best_rows[columns]
        .sort_values(metric, ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
# print(all_results.columns)
# print(all_results.to_markdown())

age_best = domain_best_by_model(
    results_df=all_results,
    domain="age",
    metric="test_accuracy_mean",
)

age_best

# print(pd.DataFrame(load_configs()))



,model_name,experiment,group,domain,test_accuracy_mean,test_f1_mean
0,random_forest,fe11__age_bin__random_forest,fe11__age_bin,age,0.833,0.759
1,svc,cb02__age_imputed_title_and_bins__svc,cb02__age_imputed_title_and_bins,age,0.832,0.767
2,xgb,cb02__age_imputed_title_and_bins__xgb,cb02__age_imputed_title_and_bins,age,0.832,0.762
3,extra_trees,fe11__age_bin__extra_trees,fe11__age_bin,age,0.819,0.737
4,knn,fe11__age_bin__knn,fe11__age_bin,age,0.810,0.737
5,decision_tree,fe11__age_bin__decision_tree,fe11__age_bin,age,0.809,0.722
6,logreg,cb03__age_imputed_title_Pclass_and_bins__logreg,cb03__age_imputed_title_Pclass_and_bins,age,0.802,0.730


In [37]:
def domain_best_by_model_with_baseline_delta(
    results_df,
    domain,
    metric="test_accuracy_mean",
):
    best_df = domain_best_by_model(
        results_df=results_df,
        domain=domain,
        metric=metric,
        include_baseline=True,
    )

    baseline = (
        results_df[results_df["group"] == "baseline__raw"]
        [["model_name", metric]]
        .rename(columns={metric: f"{metric}_baseline"})
    )

    best_df = best_df.merge(baseline, on="model_name", how="left")
    best_df[f"{metric}_delta_vs_baseline"] = (
        best_df[metric] - best_df[f"{metric}_baseline"]
    )

    return best_df

In [38]:
age_best_delta = domain_best_by_model_with_baseline_delta(
    results_df=all_results,
    domain="age",
    metric="test_accuracy_mean",
)

age_best_delta

,model_name,experiment,group,domain,test_accuracy_mean,test_f1_mean,test_accuracy_mean_baseline,test_accuracy_mean_delta_vs_baseline
0,random_forest,fe11__age_bin__random_forest,fe11__age_bin,age,0.833,0.759,0.822,0.011
1,svc,cb02__age_imputed_title_and_bins__svc,cb02__age_imputed_title_and_bins,age,0.832,0.767,0.827,0.005
2,xgb,cb02__age_imputed_title_and_bins__xgb,cb02__age_imputed_title_and_bins,age,0.832,0.762,0.826,0.006
3,extra_trees,fe11__age_bin__extra_trees,fe11__age_bin,age,0.819,0.737,0.804,0.015
4,knn,fe11__age_bin__knn,fe11__age_bin,age,0.810,0.737,0.809,0.001
5,decision_tree,fe11__age_bin__decision_tree,fe11__age_bin,age,0.809,0.722,0.803,0.006
6,logreg,cb03__age_imputed_title_Pclass_and_bins__logreg,cb03__age_imputed_title_Pclass_and_bins,age,0.802,0.730,0.786,0.016
